In [1]:
import numpy as np
import pandas as pd
import os
import re
from pathlib import Path

Function to clean the text from text files using RegEx.

In [2]:
def clean_text(text):
    # split into lines and drop first line
    lines = text.splitlines()[1:]
    text = " ".join(lines)
    text = re.sub(r"\d+Embed", "", text)

    # remove square bracket content
    text = re.sub(r"\[.*?\]", "", text)

    # remove anything that isn't a word or number
    text = re.sub(r"[^A-Za-z0-9' ]", "", text)
    # normalise whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

Read data from files.

In [3]:
DATA = Path("../data/raw/Albums")
folders = [
    'TaylorSwift', 
    'Fearless_TaylorsVersion_', 
    'SpeakNow_TaylorsVersion_',
    'Red_TaylorsVersion_',
    '1989_TaylorsVersion__Deluxe_',
    'Reputation',
    'Lover',
    'Folklore',
    'evermore_deluxeversion_',
    'Midnights_TheTilDawnEdition_',
    'THETORTUREDPOETSDEPARTMENT_THEANTHOLOGY',
    'thelakes_7_Single_RecordStoreDayExclusive_' #'the lakes'
] 

corpus = []
names = []
for folder in folders:
    for filename in sorted(os.listdir(DATA / folder)): 
        if filename.endswith(".txt"):
            filepath = DATA / folder / filename
            names.append(filename.split('.')[0]) # drop .txt from the end
            with open(filepath, "r", encoding="utf-8") as f:
                corpus.append(clean_text(f.read()))
len(corpus), names[0] # Extra songs will be dropped upon pd.merge

(246, 'APerfectlyGoodHeart')

In [4]:
corpus[0]

"Why would you wanna break A perfectly good heart Why would you wanna take Our love and tear it all apart now Why would you wanna make The very first scar Why would you wanna break A perfectly good heart Maybe I should've seen the signs Should've read the writing on the wall And realized by the distance in your eyes That I would be the one to fall No matter what you say I still can't believe that you would walk away It don't make sense to me but Why would you wanna break A perfectly good heart Why would you wanna take Our love and tear it all apart now Why would you wanna make The very first scar Why would you wanna break A perfectly good heart It's not unbroken anymore It's not unbroken anymore How do I get it back the way it was before Why would you wanna break A perfectly good heart Why would you wanna take Our love and tear it all apart now Why would you wanna make The very first scar Why would you wanna break Why Would you wanna break it You might also like Why would you wanna bre

Other embedding methods were considered (tf-idf), but Sentence Transformers are more effective at keeping semantics, especially with lyrics/poetry. 

Version 5.4.1 is used.

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(corpus, normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

This transformer gives embeddings in 768 dimensions. To avoid effects of the curse of dimensionality, PCA should be used to isolate the most important dimensions.

In [6]:
from sklearn.decomposition import PCA
pca = PCA()
pca.fit(embeddings)

PCA()

In [7]:
eigenvalues = pca.explained_variance_
cum = np.cumsum(eigenvalues)/sum(eigenvalues)
cum[39] # 40 dimensions

np.float32(0.645881)

40 dimensions retains around 65% of the variance. With under 250 observations, this seems like a reasonable balance. 

Increasing dimensions would increase explained variance, but too many dimensions leads to data points being close to equally distant to each other.

In [8]:
X = PCA(n_components=40).fit_transform(embeddings)

df = pd.DataFrame(X, columns=[f'Component_{n}' for n in range(1, 41)])
df['name'] = names
df.to_csv('lyrics_transformed.csv')